# Inference and Autoregressive Generation

> **Previous recap**: after training, given a token sequence the model outputs logits at each position (predictions for the next token).
>
> **Now the question**: how do we make it actually *talk*?
>
> Goal for this part: **understand autoregressive generation and implement greedy decoding, sampling, and temperature, then run the full pipeline end to end.**

This is the last step of the course: once you finish it, you have a mini LLM that can generate text from scratch.


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math

torch.manual_seed(42)

### 1. The fundamental difference between training and inference

Put this first:

```
Training:   answers exist -> compute loss in parallel -> teacher forcing
Inference:  no answers -> must generate token by token -> autoregressive
```

**Autoregressive** means: use the model's own generated output as the next-step input.

Example:

```
Step 1: input [BOS]              -> model predicts -> I
Step 2: input [BOS, I]           -> model predicts -> love
Step 3: input [BOS, I, love]     -> model predicts -> you
Step 4: input [BOS, I, love, you]-> model predicts -> EOS -> stop

Final output: "I love you"
```

Like a snake eating its own tail: the sequence grows by feeding itself.


### 2. The simplest generation: greedy decoding

At each step, pick the highest-probability token.
Simple and deterministic, but it often repeats.


In [ ]:
def generate_greedy(model, input_ids, max_new_tokens=20, eos_id=None):
    """ greedygeneration： token : model: training input_ids: token [1, seq_len] max_new_tokens: generation token eos_id: ID( ) : token ( ) """
    model.eval()
    generated = input_ids.clone()
    
    with torch.no_grad():
        for _ in range(max_new_tokens):
            # 1. 
            logits = model(generated)  # [1, current_len, vocab_size]
            
            # 2. logits
            next_logits = logits[0, -1, :]  # [vocab_size]
            
            # 3. , 
            probs = F.softmax(next_logits, dim=-1)
            next_token = torch.argmax(probs, dim=-1, keepdim=True)  # [1]
            
            # 4. 
            generated = torch.cat([generated, next_token.unsqueeze(0)], dim=1)
            
            # 5. EOS, 
            if eos_id is not None and next_token.item() == eos_id:
                break
    
    return generated

print("greedy generation ！")

### 3. Train a tiny model on an easy pattern

To make results visible quickly, we use an extremely simple synthetic dataset: a repeating pattern.
The model will learn fast, and we can see generation quality change.


In [ ]:
#  training ： 
#  「a b c d」 
def make_pattern_data(num_samples=200, pattern_length=8):
    """generation token """
    data = []
    for i in range(num_samples):
        seq = [(i + j) % pattern_length + 1 for j in range(pattern_length)]
        data.append(seq)
    return torch.tensor(data)

#  : 0=PAD, 1-8 token
VOCAB_SIZE = 10
SEQ_LEN = 8

train_data = make_pattern_data(200, SEQ_LEN)
print(f"training : {train_data.shape}")
print(f" 3 :")
print(train_data[:3])
print(f"\n : 0->1, 1->2, 2->3, ..., 7->8, 8->1")

In [ ]:
#  Part 5 MiniGPT( PyTorch MultiheadAttention )
import torch.nn as nn

class SimpleGPT(nn.Module):
    def __init__(self, vocab_size, d_model=32, num_heads=2, num_layers=2):
        super().__init__()
        self.d_model = d_model
        self.token_emb = nn.Embedding(vocab_size, d_model)
        self.pos_emb = nn.Embedding(64, d_model)  #  
        self.blocks = nn.ModuleList([
            nn.TransformerEncoderLayer(
                d_model=d_model, nhead=num_heads, dim_feedforward=4*d_model,
                batch_first=True, activation='relu'
            )
            for _ in range(num_layers)
        ])
        self.lm_head = nn.Linear(d_model, vocab_size)
    
    def forward(self, x):
        batch, seq = x.shape
        positions = torch.arange(seq, device=x.device).unsqueeze(0).expand(batch, -1)
        x = self.token_emb(x) + self.pos_emb(positions)
        mask = nn.Transformer.generate_square_subsequent_mask(seq, device=x.device)
        for block in self.blocks:
            x = block(x, src_mask=mask, is_causal=True)
        return self.lm_head(x)

print("SimpleGPT ！")

In [ ]:
# training
model = SimpleGPT(VOCAB_SIZE, d_model=32, num_heads=2, num_layers=2)
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)

BATCH_SIZE = 16
NUM_EPOCHS = 30

print(f"training {NUM_EPOCHS} epoch...")
model.train()
for epoch in range(NUM_EPOCHS):
    total_loss = 0
    for i in range(0, len(train_data), BATCH_SIZE):
        batch = train_data[i:i+BATCH_SIZE]
        input_ids = batch[:, :-1]
        targets = batch[:, 1:]
        
        logits = model(input_ids)
        loss = F.cross_entropy(logits.reshape(-1, VOCAB_SIZE), targets.reshape(-1))
        
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    
    if (epoch + 1) % 5 == 0:
        print(f"Epoch {epoch+1:3d} | Loss: {total_loss:.4f}")

print("training ！")

### 4. Generate: can the model continue the pattern?

The pattern is `0->1->2->...->8->1`.
Give the model a prefix and see whether it can continue.


In [ ]:
#  generation
#  [1, 2], generation 3, 4, 5, ...
prompt = torch.tensor([[1, 2]])  #  3, 4, 5, 6, 7, 8

print(f"Prompt: {prompt.tolist()}")

# Greedy generation
result = generate_greedy(model, prompt, max_new_tokens=15)
print(f"generation : {result.tolist()}")

#  
expected = [1, 2, 3, 4, 5, 6, 7, 8, 1, 2, 3, 4, 5, 6, 7, 8]
print(f" : {expected[:len(result[0])]}")

correct = sum(1 for a, b in zip(result[0].tolist(), expected) if a == b)
print(f" {len(result[0])} token : {correct}/{len(result[0])}")

### 5. Try different prompts

Give different prefixes to confirm the model learned the rule, not just memorized one case.


In [ ]:
test_prompts = [
    [5],          #  : 5->6->7->8->1->2->...
    [3, 4],       #  : 3->4->5->6->...
    [8],          #  : 8->1->2->3->... (wraparound)
]

for prompt_tokens in test_prompts:
    prompt = torch.tensor([prompt_tokens])
    result = generate_greedy(model, prompt, max_new_tokens=10)
    print(f" {prompt_tokens} -> generation {result[0].tolist()}")

### 6. Temperature: control "creativity"

Greedy decoding is too rigid: it always picks argmax, so you get the same output every time.

Real LLMs use **temperature sampling** for diversity:

```
probs = softmax(logits / temperature)

temperature = 0.1 -> sharper peak -> almost greedy (conservative)
temperature = 1.0 -> original distribution
temperature = 2.0 -> flatter distribution -> more random (more creative but riskier)
```

Intuition: lower temperature = more confident; higher temperature = more random.


In [ ]:
#  temperature 
import matplotlib.pyplot as plt

#  logits
example_logits = torch.tensor([0.5, 2.0, 1.0, 0.1, 0.3])

fig, axes = plt.subplots(1, 4, figsize=(14, 3))
temperatures = [0.2, 0.7, 1.5, 5.0]

for ax, T in zip(axes, temperatures):
    probs = F.softmax(example_logits / T, dim=-1)
    ax.bar(range(len(probs)), probs.numpy())
    ax.set_title(f'T={T}')
    ax.set_ylim(0, 1)
    ax.set_xlabel('token ID')
    if T == 0.2:
        ax.set_ylabel('Probability')

plt.suptitle('Temperature ')
plt.tight_layout()
plt.show()

print("T=0.2: token -> / ")
print("T=1.5: -> ")
print("T=5.0: -> ")


### 7. A generation function with temperature (and top-k)

Upgrade the generator to support temperature and top-k sampling.


In [ ]:
def generate_with_temperature(model, input_ids, max_new_tokens=20, 
                              temperature=1.0, top_k=None, eos_id=None):
    """ temperature top-k autoregressivegeneration : temperature: ,<1 ,>1 top_k: k token sampling """
    model.eval()
    generated = input_ids.clone()
    
    with torch.no_grad():
        for _ in range(max_new_tokens):
            logits = model(generated)
            raw_logits = logits[0, -1, :]

            if temperature <= 0:
                next_token = torch.argmax(raw_logits).unsqueeze(0)
                generated = torch.cat([generated, next_token.unsqueeze(0)], dim=1)
                if eos_id is not None and next_token.item() == eos_id:
                    break
                continue

            next_logits = raw_logits / temperature
            
            # Top-k: k 
            if top_k is not None:
                top_k_values, _ = torch.topk(next_logits, top_k)
                min_top_k = top_k_values[-1]
                next_logits[next_logits < min_top_k] = float('-inf')
            
            #  + sampling
            probs = F.softmax(next_logits, dim=-1)
            next_token = torch.multinomial(probs, num_samples=1)
            
            generated = torch.cat([generated, next_token.unsqueeze(0)], dim=1)
            
            if eos_id is not None and next_token.item() == eos_id:
                break
    
    return generated

print(" temperature generation ！")

In [ ]:
#  temperature generation 
prompt = torch.tensor([[2]])

print(" prompt [2], temperature:")
print()

for T in [0.1, 0.5, 1.0, 2.0]:
    torch.manual_seed(123)  #  
    result = generate_with_temperature(model, prompt, max_new_tokens=10, temperature=T)
    print(f"T={T:.1f}: {result[0].tolist()}")

print(f"\nT=0.1 ( ) -> -> training ")
print(f"T=2.0 ( ) -> -> ")

### 8. Top-k sampling: avoid low-probability junk tokens

Even with temperature=1.0, many tokens have extremely low probability. If you sample them, quality drops.

Top-k sampling keeps only the top k tokens and renormalizes:

```
All token probs: [0.01, 0.50, 0.30, 0.10, 0.05, 0.02, ...]
Top-k=3: keep [0.50, 0.30, 0.10] and renormalize
```


In [ ]:
#  top-k 
prompt = torch.tensor([[2]])

print(" prompt [2], top_k:")
print()

for k in [1, 3, 5, None]:
    torch.manual_seed(42)
    result = generate_with_temperature(
        model, prompt, max_new_tokens=10, temperature=0.7, top_k=k
    )
    label = f"top_k={k}" if k else "no top_k"
    
    #  
    tokens = result[0].tolist()
    ok = all(tokens[i] == (tokens[i-1] % 8) + 1 for i in range(2, len(tokens)) if tokens[i-1] <= 8)
    status = "[ok] " if ok else "⚠️ "
    
    print(f"{label:12s}: {tokens} {status}")

print(f"\ntop_k=1 = greedy( )")
print(f"top_k=3 = ")
print(f"no top_k = , token")

### 9. Full pipeline: from text to text

Connect everything from Part 1 to Part 6:

```
User text
  -> Tokenizer.encode()            (Parts 1-2)
  -> token IDs
  -> TokenEmbedding + Position     (Part 3)
  -> vectors
  -> N x Transformer blocks        (Part 4)
  -> logits
  -> training uses CrossEntropy    (Part 5)

After training...

Prompt (token IDs)
  -> autoregressive generation     (Part 6)
  -> generated token IDs
  -> Tokenizer.decode()            (Parts 1-2)
  -> output text
```

This is the full LLM pipeline. GPT-4/Claude/Gemini are bigger versions of the same idea.


### 10. Top-p (nucleus sampling): smarter truncation than top-k

Top-k uses a fixed number k.
If the distribution is very peaked, k may include junk tokens.
If the distribution is very flat, k may exclude many reasonable options.

Top-p uses a probability mass threshold p:
accumulate from highest prob until the sum >= p.

Example:

```
probs: [0.4, 0.3, 0.15, 0.08, 0.04, 0.02, 0.01]

Top-k=3: 0.4+0.3+0.15 = 0.85
Top-p=0.9: 0.4+0.3+0.15+0.08 = 0.93 -> stop
```

Top-p adapts: peaked -> fewer tokens, flat -> more tokens.


In [ ]:
#  top-p (nucleus) sampling
def top_p_filter(logits, p=0.9):
    sorted_logits, sorted_indices = torch.sort(logits, descending=True)
    cumulative_probs = torch.cumsum(torch.softmax(sorted_logits, dim=-1), dim=-1)
    sorted_indices_to_remove = cumulative_probs > p
    sorted_indices_to_remove[..., 0] = False
    indices_to_remove = sorted_indices[sorted_indices_to_remove]
    logits[:, indices_to_remove] = float('-inf')
    return logits

print('=== top-k vs top-p ===')
prompt = torch.tensor([[2]])
with torch.no_grad():
    logits = model(prompt)[0, -1, :]
probs = torch.softmax(logits, dim=-1)
sorted_probs, sorted_idx = torch.sort(probs, descending=True)
print(' ( ):')
for i in range(10):
    print(f'  token {sorted_idx[i].item()}: {sorted_probs[i].item():.4f}')
print()
print('top-k (k=3) token:', sorted_idx[:3].tolist())
cumsum = torch.cumsum(sorted_probs, dim=-1)
n_top_p = (cumsum <= 0.9).sum().item() + 1
print(f'top-p (p=0.9) token: {sorted_idx[:n_top_p].tolist()} ( {n_top_p} )')
print()
print('Key ：top-k ,top-p .')

### 11. Beam search: keep multiple paths

Greedy is locally optimal, not globally optimal.

Beam search keeps K partial hypotheses (beam size).
At each step, expand all beams and keep the best K by total score.

Best for tasks with a clear correct target (translation, summarization).
Often bad for creative writing and chat (outputs become bland).


In [ ]:
def beam_search(model, input_ids, beam_size=3, max_new_tokens=10, eos_id=None):
    batch_size = input_ids.shape[0]
    beam_scores = torch.zeros(batch_size, beam_size)
    beam_sequences = input_ids.repeat(beam_size, 1)
    beam_done = [False] * beam_size
    for step in range(max_new_tokens):
        if all(beam_done):
            break
        with torch.no_grad():
            logits = model(beam_sequences)
        next_log_probs = torch.log_softmax(logits[:, -1, :], dim=-1)
        cumulative_scores = beam_scores[:, :1] + next_log_probs
        flat_scores = cumulative_scores.view(-1)
        top_scores, top_indices = torch.topk(flat_scores, beam_size)
        beam_indices = top_indices // logits.shape[-1]
        token_indices = top_indices % logits.shape[-1]
        new_beam_sequences, new_beam_scores, new_beam_done = [], [], []
        for i in range(beam_size):
            b_idx = beam_indices[i].item()
            t_idx = token_indices[i].item()
            if beam_done[b_idx]:
                new_beam_sequences.append(beam_sequences[b_idx])
                new_beam_scores.append(beam_scores[0, b_idx])
                new_beam_done.append(True)
            else:
                new_seq = torch.cat([beam_sequences[b_idx], token_indices[i].unsqueeze(0)])
                new_beam_sequences.append(new_seq)
                new_beam_scores.append(top_scores[i])
                new_beam_done.append(eos_id is not None and t_idx == eos_id)
        beam_sequences = torch.stack([s[:beam_sequences.shape[1]+1] for s in new_beam_sequences])
        beam_scores = torch.tensor(new_beam_scores).unsqueeze(0)
        beam_done = new_beam_done
    best_idx = beam_scores[0].argmax().item()
    return beam_sequences[best_idx].unsqueeze(0)

print('Beam Search ！')
prompt = torch.tensor([[2]])
greedy_result = generate_with_temperature(model, prompt, max_new_tokens=8, temperature=0.0)
print(f'Greedy:      {greedy_result[0].tolist()}')
beam_result = beam_search(model, prompt, beam_size=3, max_new_tokens=8)
print(f'Beam (k=3):  {beam_result[0].tolist()}')

### 12. Repetition penalty: stop the model from becoming a parrot

LLMs often get stuck repeating phrases.

A repetition penalty reduces the logits of tokens that already appeared, breaking the feedback loop.


In [ ]:
def apply_repetition_penalty(logits, generated_ids, penalty=1.2):
    if penalty == 1.0:
        return logits
    for token_id in set(generated_ids.tolist()):
        score = logits[:, token_id]
        logits[:, token_id] = torch.where(score > 0, score / penalty, score * penalty)
    return logits

print('=== Repetition Penalty ===')
prompt = torch.tensor([[2]])
for penalty in [1.0, 1.2, 1.5]:
    generated = prompt.clone()
    with torch.no_grad():
        for _ in range(8):
            logits = model(generated)[0, -1, :].unsqueeze(0)
            logits = apply_repetition_penalty(logits, generated[0], penalty=penalty)
            probs = torch.softmax(logits / 0.7, dim=-1)
            next_token = torch.multinomial(probs, 1)
            generated = torch.cat([generated, next_token], dim=1)
    tokens = generated[0].tolist()
    unique_ratio = len(set(tokens)) / len(tokens)
    print(f'penalty={penalty:.1f}: {tokens} ( ={unique_ratio:.1%})')
print()
print('penalty , . 1.1~1.2 .')

### 13. Stopping criteria: when to stop generation

Common stopping criteria:

| Condition | Meaning |
|:---|:---|
| EOS token | model produced `<EOS>` -> natural stop |
| max_new_tokens | reached max length -> forced stop |
| repeated tokens | detect loops -> stop |
| n-gram repetition | repeated n-grams too many times -> stop |

EOS is the most natural stop. That is why tokenizers define an EOS special token.


In [ ]:
def generate_full(model, input_ids, max_new_tokens=50, temperature=1.0,
                  top_k=None, top_p=None, repetition_penalty=1.0, eos_id=None):
    generated = input_ids.clone()
    for _ in range(max_new_tokens):
        with torch.no_grad():
            logits = model(generated)[0, -1, :].unsqueeze(0)
        if repetition_penalty != 1.0:
            logits = apply_repetition_penalty(logits, generated[0], repetition_penalty)
        logits = logits / temperature
        if top_k is not None:
            top_k_values, _ = torch.topk(logits, top_k)
            logits[logits < top_k_values[:, -1:]] = float('-inf')
        if top_p is not None:
            logits = top_p_filter(logits, top_p)
        probs = torch.softmax(logits, dim=-1)
        next_token = torch.multinomial(probs, 1)
        if eos_id is not None and next_token.item() == eos_id:
            break
        generated = torch.cat([generated, next_token], dim=1)
    return generated

print(' generation ！')
print()
print(' ( ):')
print(' 1. Repetition Penalty -> ')
print(' 2. Temperature -> ')
print(' 3. Top-k / Top-p -> token')
print(' 4. Sampling -> sampling')
print(' 5. EOS check -> ')

---

## Part 6 Summary + Course Recap

### Part 6 checklist

1. [ok] Training vs inference: training has answers (parallel), inference does not (serial)
2. [ok] Autoregressive: feed generated tokens back as input
3. [ok] Greedy: argmax each step
4. [ok] Temperature: controls randomness
5. [ok] Top-k: limit candidate tokens
6. [ok] Top-p: truncate by probability mass
7. [ok] Beam search: maintain multiple hypotheses
8. [ok] Repetition penalty: reduce logits for seen tokens
9. [ok] Stopping criteria: EOS / max length / repetition checks

### Full course recap

| Part | What you learned | Industrial concept |
|:---|:---|:---|
| 1 Tokenizer | char/word/subword tradeoffs | BPE/WordPiece/SentencePiece |
| 2 BPE | merges, encode, decode | GPT-2 BPE files |
| 3 Embedding | IDs -> vectors + position | nn.Embedding + positional encoding |
| 4 Attention | Q/K/V + softmax + multi-head | scaled dot-product attention |
| 5 Loss | teacher forcing + cross-entropy | how all autoregressive LMs train |
| 6 Generate | autoregressive + sampling | how all LLMs run inference |

Next ideas:
1. train on real text and see style emerge
2. plug in a real tokenizer (tiktoken)
3. implement KV cache to speed up decoding
4. scale up d_model/layers
5. implement RoPE

Congrats: you now understand a full mini-LLM pipeline end to end.
